In [36]:
import pandas as pd 
import numpy as np

In [37]:
df=pd.read_parquet(r"C:\Users\hp\Desktop\B-tech\PYTHON_CODING\PROJECTS\FINANCIAL RISK PLATFORM\data\model_data.parquet")

In [38]:
df.head()

,ticker,date,adj_close,close,high,low,open,volume,daily_return,log_return,...,consumer_vs_inflation,growth_minus_inflation,retail_vs_unemployment,gdp_up,inflation_up,rates_up,economic_stress,yield_curve_change,yield_curve_ma6,future_return
0,AAPL,2018-02-07,37.293663,39.884998,40.849998,39.767502,40.772499,206434400,-0.021407,-0.021640,...,0.399553,20079.024,118056.341463,0.0,0.0,0.0,1.132843,0.05,1.246667,-0.027517
1,AAPL,2018-02-08,36.267479,38.787498,40.250000,38.757500,40.072498,217562000,-0.027517,-0.027902,...,0.399553,20079.024,118056.341463,0.0,0.0,0.0,0.957427,0.01,1.241667,0.008121
2,AAPL,2018-02-09,36.711082,39.102501,39.472500,37.560001,39.267502,282690400,0.008121,0.008088,...,0.399553,20079.024,118056.341463,0.0,0.0,0.0,0.809174,-0.02,1.250000,0.040279
3,AAPL,2018-02-12,38.189770,40.677502,40.972500,39.377499,39.625000,243278000,0.040279,0.039489,...,0.399553,20079.024,118056.341463,0.0,0.0,0.0,0.677003,0.03,1.253333,0.010018
4,AAPL,2018-02-13,38.572338,41.084999,41.187500,40.412498,40.487499,130196800,0.010018,0.009968,...,0.399553,20079.024,118056.341463,0.0,0.0,0.0,0.552771,-0.03,1.263333,0.018437


In [39]:
df.shape

(101280, 270)

In [40]:
df = df.sort_values(["date", "ticker"]).reset_index(drop=True)
df = df.dropna().reset_index(drop=True)


In [41]:
print(df.columns.to_list())

['ticker', 'date', 'adj_close', 'close', 'high', 'low', 'open', 'volume', 'daily_return', 'log_return', 'return_5d', 'return_10d', 'return_20d', 'return_60d', 'gap', 'open_close_diff', 'high_low_spread', 'price_range_pct', 'sma_5', 'sma_20', 'sma_50', 'ema_12', 'ema_26', 'rsi_14', 'roc_10', 'dist_sma5', 'dist_sma20', 'dist_sma50', 'dist_ema12', 'dist_ema26', 'ema_diff', 'ema_ratio', 'rolling_std_5', 'rolling_std_10', 'rolling_std_20', 'rolling_std_60', 'ewma_vol_20', 'volume_sma20', 'relative_volume', 'rolling_sharpe_20', 'price_position_20', 'volume_zscore', 'rsi_above70', 'rsi_below30', 'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_middle', 'bb_lower', 'bb_width', 'adx_14', 'drawdown', 'atr_14', 'obv', 'news_count', 'avg_sentiment', 'max_sentiment', 'min_sentiment', 'std_sentiment', 'avg_confidence', 'positive_count', 'negative_count', 'neutral_count', 'positive_ratio', 'negative_ratio', 'neutral_ratio', 'sentiment_3d', 'sentiment_7d', 'sentiment_14d', 'news_count_3d', 'news_co

In [42]:
df.columns

Index(['ticker', 'date', 'adj_close', 'close', 'high', 'low', 'open', 'volume',
       'daily_return', 'log_return',
       ...
       'consumer_vs_inflation', 'growth_minus_inflation',
       'retail_vs_unemployment', 'gdp_up', 'inflation_up', 'rates_up',
       'economic_stress', 'yield_curve_change', 'yield_curve_ma6',
       'future_return'],
      dtype='object', length=270)

In [32]:
import numpy as np

df["y"] = np.where(
    df["future_return"] * 100 >= 1,
    "buy",
    "don't buy"
)

In [43]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

df["future_volatility"] = (
    df.groupby("ticker")["future_return"]
      .transform(lambda x: x.rolling(5).std().shift(-4))
)
df["risk_category"] = pd.qcut(df["future_volatility"],q=4,labels=["Low","Medium","High","Very High"])
df = df.dropna(subset=["future_volatility", "risk_category"]).reset_index(drop=True)

x = df.drop(columns=["future_return","ticker","date","risk_category","future_volatility"])
y = pd.Series(
    le.fit_transform(df["risk_category"]),
    index=df.index,
    name="risk_category"
)
df = df.dropna().reset_index(drop=True)
split = int(len(df) * 0.8)
split = int(len(df) * 0.8)
x_train = x.iloc[:split]
x_test = x.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]



In [ ]:
x=df.drop(columns=['future_return','y','ticker','date'])
y=df['y']
split = int(len(df) * 0.8)
x_train, x_test = x.iloc[:split], x.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]


In [47]:
import joblib
model=joblib.load(r"C:\Users\hp\Desktop\B-tech\PYTHON_CODING\PROJECTS\FINANCIAL RISK PLATFORM\models\best_classifier_model.pkl")
print(model)

CatBoostClassifier(depth=4, iterations=500, l2_leaf_reg=3, learning_rate=0.01, loss_function='MultiClass', random_seed=42, verbose=0)


In [45]:
print(score)

0.390196854288258


In [17]:
!pip install catboost

  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
    --------------------------------------- 2.1/100.2 MB 7.8 MB/s eta 0:00:13
   -- ------------------------------------- 5.2/100.2 MB 10.7 MB/s eta 0:00:09
   -- ------------------------------------- 6.0/100.2 MB 10.6 MB/s eta 0:00:09
   ---- ----------------------------------- 10.5/100.2 MB 11.6 MB/s eta 0:00:08
   ---- ----------------------------------- 11.8/100.2 MB 10.6 MB/s eta 0:00:09
   ------ --------------------------------- 15.5/100.2 MB 11.5 MB/s eta 0:00:08
   ------- -------------------------------- 18.4/100.2 MB 11.7 MB/s eta 0:00:07
   ------- -------------------------------- 19.9/100.2 MB 11.8 MB/s eta 0:00:07
   --------- ------------------------------ 23.1/100.2 MB 11.7 MB/s eta 0:00:07
   -----


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
